# Time-Series Anomaly Detection Engine
### A three-tier pipeline: Statistical → Classic ML → Deep Learning

This notebook implements the revised project blueprint end-to-end. The design philosophy is to build
**three escalating detectors** that mirror the evolution of time-series modelling, and to do it under a
**leakage-free, time-based evaluation** so the results survive interview scrutiny.

| Tier | Detector | Family | What it's good at | Blind spot |
|------|----------|--------|-------------------|------------|
| 1 | Trailing Z-Score | Statistical | Sharp, instant spikes; ~zero compute | No multi-feature view |
| 2 | Isolation Forest | Classic ML | Multi-dimensional outliers | **Ignores temporal order** |
| 3 | LSTM Autoencoder | Deep learning | Slow structural shifts via memory | Needs lots of data + tuning |

**Design decisions baked in (the things an interviewer probes):**
1. **Split by time, fit every scaler on the training window only** — no future leakage.
2. **Enough data for the LSTM** — pull years of history, not a few hundred rows.
3. **Train the autoencoder on a normal period only** — so anomalies actually produce high reconstruction error.
4. **Real validation** — inject synthetic anomalies to get precision/recall, and cross-check against known events.
5. **Polish** — trailing (causal) z-score, log-transformed volume, model on *returns* not raw price, fully seeded runs.

> Run top-to-bottom in Google Colab. The only cell that needs the internet is the `yfinance` download.

## Setup — installs, imports, and global seed

We seed Python, NumPy and TensorFlow so the run is reproducible (the LSTM in particular is stochastic).

In [ ]:
# Colab usually has tensorflow / sklearn / pandas preinstalled; yfinance & plotly may not be.
!pip install -q yfinance plotly --upgrade

In [ ]:
import os, random
import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score

import tensorflow as tf
from tensorflow.keras import layers, Sequential

# ---- Global reproducibility ----
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("TensorFlow:", tf.__version__)
print("Environment ready. Seed =", SEED)

## Configuration

Everything you might tune lives here. Defaults follow the blueprint: a liquid, volatile asset and a long
daily history so the deep tier has thousands of points to learn from.

In [ ]:
CONFIG = {
    # --- Phase 1: data ---
    # RELIANCE.NS gives ~decades of daily history -> plenty for the LSTM.
    # Alternatives: "^NSEI" (Nifty 50 index). Avoid young tickers like "ZOMATO.NS"
    # (IPO 2021) for the LSTM tier -- too few rows, the autoencoder overfits.
    "TICKER": "RELIANCE.NS",
    "PERIOD": "10y",        # yfinance limits: minute~7d, hourly~730d, daily~decades
    "INTERVAL": "1d",

    # --- Phase 2: features / split ---
    "VOL_WINDOW": 20,       # rolling volatility window (trading days)
    "TRAIN_FRAC": 0.70,     # earlier 70% = mostly-normal training period

    # --- Phase 3: detectors ---
    "Z_WINDOW": 20,         # trailing z-score window
    "Z_THRESH": 3.0,        # |z| > 3 -> anomaly
    "CONTAMINATION": 0.03,  # Isolation Forest: ~3% expected anomalies (a HYPERPARAMETER, justify it)
    "SEQ_LEN": 10,          # LSTM sequence window (days)
    "AE_PCTILE": 99,        # threshold = this percentile of TRAIN reconstruction error
    "EPOCHS": 50,
    "BATCH": 64,

    # --- Phase 4: validation ---
    "N_INJECT": 12,         # synthetic anomalies to inject
    "INJECT_RET_MULT": 6.0, # how hard to spike the return
    "INJECT_VOL_MULT": 5.0, # how hard to spike the volume
    "MATCH_TOL": 1,         # +/- days tolerance when matching a flag to an injected date

    "INJECT_VOLZ": 4.0,     # injected volume-zscore jump (std units), for the LSTM channel

    # Isolation Forest sees the full multi-dim view -- it handles the volume trend fine.
    "FEATURES_ISO":  ["Return", "Volatility", "LogVolume"],
    # The LSTM must see ONLY stationary features. Raw LogVolume trends upward over the years,
    # so the autoencoder would flag "recent era" as anomalous (regime drift) instead of true
    # anomalies. VolZScore = trailing z-score of log-volume keeps the volume signal, stays stationary.
    "FEATURES_LSTM": ["Return", "Volatility", "VolZScore"],
}
CONFIG

---
## Phase 1 · Data Strategy & Setup
**Goal:** isolate the genuine irregularities in the series — the rare points the market's normal behaviour doesn't explain.

### Step 1 — Select a liquid, volatile asset
We use a highly liquid Indian equity (**Reliance**) with real chaos for the detectors to find.

> **Talking point:** in a volatile asset, *large moves are partly normal*. That makes "anomaly" genuinely hard
> to define — which is exactly the kind of nuance worth raising in a write-up.

### Step 2 — Pull enough data
`yfinance` for **Close** and **Volume**. We pull years of daily data so the LSTM tier sees *thousands* of points.

> **Why this matters:** 1–2 years of daily data is only ~250–500 rows — an LSTM autoencoder overfits badly on that.
> Longer daily history is the simplest fix.

In [ ]:
import yfinance as yf

raw = yf.download(
    CONFIG["TICKER"],
    period=CONFIG["PERIOD"],
    interval=CONFIG["INTERVAL"],
    auto_adjust=True,
    progress=False,
)

# yfinance can return MultiIndex columns for a single ticker -> flatten.
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = raw[["Close", "Volume"]].copy()
df.index = pd.to_datetime(df.index)
print(f"Pulled {len(df):,} rows for {CONFIG['TICKER']}  "
      f"({df.index.min().date()} -> {df.index.max().date()})")
df.head()

### Step 3 — Handle gaps carefully
Don't blindly drop rows — that silently breaks the even time spacing the models assume.

- **Weekend / holiday gaps are expected** → leave them as-is (the trading calendar is the real index).
- **Stray missing values inside trading days** → *forward-fill* rather than drop.

In [ ]:
before = df.isna().sum().to_dict()

# Forward-fill stray intra-trading-day gaps; back-fill only a leading NaN if present.
df = df.ffill().bfill()

after = df.isna().sum().to_dict()
print("Missing values before:", before)
print("Missing values after :", after)
print("Note: weekend/holiday calendar gaps are intentionally preserved (not reindexed to daily).")

---
## Phase 2 · Feature Engineering & Preprocessing
**Goal:** give the models temporal context, and present the data in the form each tier needs.

### Step 4 — Create contextual features
- **Daily % Return** and **20-day Rolling Volatility** → a multi-dimensional view of momentum.
- **Log-transform Volume** → raw volume is heavily right-skewed.
- Feed **stationary** features to the LSTM rather than raw price levels (which trend and are hard to learn).
  Returns and volatility are already stationary; for volume we use **`VolZScore`** — a trailing z-score of
  log-volume — instead of raw `LogVolume`, which creeps upward over the years. (Isolation Forest still gets the
  full raw view, including `LogVolume`.)

In [ ]:
df["Return"]     = df["Close"].pct_change()
df["Volatility"]  = df["Return"].rolling(CONFIG["VOL_WINDOW"]).std()
df["LogVolume"]   = np.log1p(df["Volume"])          # log1p handles any zero-volume days
# Stationary volume signal for the LSTM: how unusual is today's volume vs the last ~month?
# (raw LogVolume trends up over the years and would make the autoencoder flag regime drift.)
_w = CONFIG["VOL_WINDOW"]
df["VolZScore"]   = (df["LogVolume"] - df["LogVolume"].rolling(_w).mean()) / df["LogVolume"].rolling(_w).std()

# Rolling/return calcs create NaNs at the head -> drop just those warm-up rows.
df = df.dropna().copy()
print("Feature frame:", df.shape)
df[["Close", "Return", "Volatility", "LogVolume"]].describe()

### Step 5 — Split by time, then scale on the training window only
First carve a **time-based** split (earlier 70% = mostly-normal training; later 30% held out). **Never shuffle.**
Then fit scalers on the **training data only** and apply them to everything:

- **Standardization** (mean 0, std 1) for the statistical / Isolation-Forest features.
- **Min-Max** (0–1) for the neural network — unscaled inputs make the LSTM unstable.

> **Most important fix:** fitting a scaler's mean/std or min/max on the *whole* series leaks future information
> into your evaluation. Fit on **train only**. This is the first thing a sharp interviewer probes.

In [ ]:
split = int(len(df) * CONFIG["TRAIN_FRAC"])
train_df = df.iloc[:split].copy()
test_df  = df.iloc[split:].copy()
iso_feat  = CONFIG["FEATURES_ISO"]    # full multi-dim view  -> Isolation Forest
lstm_feat = CONFIG["FEATURES_LSTM"]   # stationary-only view -> LSTM

# Fit on TRAIN ONLY -- the leakage-safety guarantee. Standardize the ISO features,
# Min-Max the (stationary) LSTM features.
std_scaler = StandardScaler().fit(train_df[iso_feat])
mm_scaler  = MinMaxScaler().fit(train_df[lstm_feat])

# Apply to both windows.
Z_train = std_scaler.transform(train_df[iso_feat]); Z_test = std_scaler.transform(test_df[iso_feat])
M_train = mm_scaler.transform(train_df[lstm_feat]);  M_test = mm_scaler.transform(test_df[lstm_feat])

print(f"Train: {len(train_df):,} rows  ({train_df.index.min().date()} -> {train_df.index.max().date()})")
print(f"Test : {len(test_df):,} rows  ({test_df.index.min().date()} -> {test_df.index.max().date()})")
print("Scaler means are computed from TRAIN ONLY (no future leakage):",
      np.round(std_scaler.mean_, 4))

---
## Phase 3 · Building the Three Tiers
Each detector is written as a reusable function so we can run it on the real data **and** reuse the exact
same code in the validation harness (Phase 4).

### Step 6 — Tier 1: the Z-Score baseline (statistical)
A **trailing** 20-day rolling mean & std; flag any point more than 3σ away. The window is *trailing, not
centered*, so it only ever looks backward — **causal and leak-free by construction**. Near-zero compute,
catches sharp spikes instantly.

In [ ]:
def zscore_detect(returns: pd.Series, window=CONFIG["Z_WINDOW"], k=CONFIG["Z_THRESH"]):
    """Causal trailing z-score on the return series. Returns (bool flags, z values)."""
    roll_mean = returns.rolling(window).mean()
    roll_std  = returns.rolling(window).std()
    z = (returns - roll_mean) / roll_std
    flags = z.abs() > k
    return flags.fillna(False), z

z_flags_all, z_vals_all = zscore_detect(df["Return"])
print(f"Z-Score flagged {int(z_flags_all.sum())} of {len(df)} days "
      f"({100*z_flags_all.mean():.2f}%)")

### Step 7 — Tier 2: Isolation Forest (classic ML)
Feed the multi-dimensional features (returns, volatility, log-volume) into the tree algorithm with an
**explicit contamination (~3%)** and a fixed seed. Treat contamination as a hyperparameter you can justify,
not a magic number — it's your prior on how often anomalies occur.

> **Worth saying out loud:** Isolation Forest scores each day as an *independent feature vector* — it ignores
> temporal order entirely. That blind spot is exactly why the LSTM tier exists.

In [ ]:
# Fit on TRAIN ONLY, then score the whole series (standardized features).
iso = IsolationForest(
    contamination=CONFIG["CONTAMINATION"],
    random_state=SEED,
    n_estimators=200,
).fit(Z_train)

iso_pred_all = iso.predict(std_scaler.transform(df[iso_feat]))      # -1 = anomaly, 1 = normal
iso_flags_all = pd.Series(iso_pred_all == -1, index=df.index)
# Lower score = more anomalous; keep it for ranking/inspection.
iso_score_all = pd.Series(iso.score_samples(std_scaler.transform(df[iso_feat])), index=df.index)

print(f"Isolation Forest flagged {int(iso_flags_all.sum())} of {len(df)} days "
      f"({100*iso_flags_all.mean():.2f}%)  | contamination={CONFIG['CONTAMINATION']}")

### Step 8 — Tier 3: the LSTM Autoencoder (deep learning)
Train a network to compress and reconstruct short sequences; points it **can't rebuild** are anomalies.

- **Train on a normal period only.** We train on the (mostly-normal) training window. If anomalies were in the
  training data, the model would learn to reconstruct them too and the signal would wash out.
- **Sequence building:** chop the timeline into overlapping windows (10 days) so the LSTM sees sequential blocks.
- **Scoring:** reconstruction error (MSE) per window.
- **Threshold principledly:** the **99th percentile of training reconstruction error** — not a hand-picked cutoff.

> **Stationary inputs only (the fix that makes this tier work).** The autoencoder is fed `Return`, `Volatility`
> and `VolZScore` — never raw `LogVolume`. Trading volume grows year over year, so raw log-volume in the test era
> sits outside the training range; the Min-Max scaler (fit on train) pushes it past 1.0, the autoencoder can't
> reconstruct inputs it never saw, and reconstruction error drifts upward across the whole recent period — the
> model ends up flagging *"this is a newer regime"* rather than *"this is an anomaly."* Using the stationary
> `VolZScore` keeps the volume signal while removing the trend, so the test error distribution matches train and
> the threshold means what it should. If your flag rate is far above ~1%, this is the first thing to check.

In [ ]:
def make_sequences(arr, seq_len=CONFIG["SEQ_LEN"]):
    """Overlapping windows: (n_rows, n_feat) -> (n_windows, seq_len, n_feat)."""
    return np.stack([arr[i:i+seq_len] for i in range(len(arr) - seq_len + 1)])

def build_lstm_autoencoder(seq_len, n_features):
    """Symmetric LSTM autoencoder: encoder -> latent vector -> RepeatVector -> decoder."""
    model = Sequential([
        layers.Input(shape=(seq_len, n_features)),
        layers.LSTM(32, activation="tanh", return_sequences=True),
        layers.LSTM(16, activation="tanh", return_sequences=False),   # latent vector
        layers.RepeatVector(seq_len),                                  # expand back over time
        layers.LSTM(16, activation="tanh", return_sequences=True),
        layers.LSTM(32, activation="tanh", return_sequences=True),
        layers.TimeDistributed(layers.Dense(n_features)),             # reconstruct each step
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

# Build sequences from MIN-MAX scaled features (train = the "normal" period).
X_train_seq = make_sequences(M_train)
X_test_seq  = make_sequences(M_test)
print("Train sequences:", X_train_seq.shape, "| Test sequences:", X_test_seq.shape)

In [ ]:
tf.random.set_seed(SEED)
ae = build_lstm_autoencoder(CONFIG["SEQ_LEN"], len(lstm_feat))
ae.summary()

early = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

history = ae.fit(
    X_train_seq, X_train_seq,            # autoencoder: input == target
    epochs=CONFIG["EPOCHS"],
    batch_size=CONFIG["BATCH"],
    validation_split=0.1,
    shuffle=True,                        # shuffling whole windows is fine (each preserves its own order)
    callbacks=[early],
    verbose=1,
)

In [ ]:
def recon_error(model, seqs):
    """Per-window mean squared reconstruction error."""
    pred = model.predict(seqs, verbose=0)
    return np.mean(np.square(seqs - pred), axis=(1, 2))

train_err = recon_error(ae, X_train_seq)
AE_THRESHOLD = np.percentile(train_err, CONFIG["AE_PCTILE"])
print(f"Autoencoder threshold = {CONFIG['AE_PCTILE']}th pct of TRAIN error = {AE_THRESHOLD:.6f}")

def lstm_flags_for(scaled_matrix, index):
    """Score a min-max-scaled feature matrix; map each window's error to its LAST day."""
    seqs = make_sequences(scaled_matrix)
    err  = recon_error(ae, seqs)
    # window i covers rows [i : i+SEQ_LEN-1]; attribute its error to the last row.
    aligned_idx = index[CONFIG["SEQ_LEN"] - 1:]
    err_s   = pd.Series(err, index=aligned_idx)
    flags_s = err_s > AE_THRESHOLD
    return flags_s, err_s

lstm_flags_test, lstm_err_test = lstm_flags_for(M_test, test_df.index)
print(f"LSTM flagged {int(lstm_flags_test.sum())} of {len(lstm_flags_test)} "
      f"scored test days ({100*lstm_flags_test.mean():.2f}%)")

---
## Phase 4 · Validation, Visual Proof & Conclusion

### Step 9 — Validate *before* you visualize
Anomaly detection is unsupervised, so we manufacture ground truth:

1. **Inject synthetic anomalies** at known dates in the held-out test window (spike the return and volume),
   then measure how many each tier recovers → **real precision / recall**.
2. **Cross-reference** flagged dates against real known events as a qualitative sanity check.

We evaluate all three tiers on the **same injected test window** for a fair comparison. A `±1 day` tolerance is
allowed when matching a flag to an injected date, because the LSTM attributes error to a *window*, which blurs
exact-day localisation.

In [ ]:
rng = np.random.default_rng(SEED)

# --- Build an injected copy of the test window (operate on RAW features, then re-scale) ---
inj_df = test_df.copy()
# avoid the very edges so a window/rolling stat exists around each injection
candidate_pos = np.arange(CONFIG["Z_WINDOW"], len(inj_df) - 1)
inject_pos = np.sort(rng.choice(candidate_pos, size=CONFIG["N_INJECT"], replace=False))
inject_dates = inj_df.index[inject_pos]

ret_loc = inj_df.columns.get_loc("Return")
vol_loc = inj_df.columns.get_loc("LogVolume")
volz_loc = inj_df.columns.get_loc("VolZScore")
# Spike returns (sign-preserving), push raw log-volume up (for the ISO channel),
# and jump the stationary volume z-score up (for the LSTM channel) -- consistently.
inj_df.iloc[inject_pos, ret_loc]  *= CONFIG["INJECT_RET_MULT"]
inj_df.iloc[inject_pos, vol_loc]  += np.log(CONFIG["INJECT_VOL_MULT"])
inj_df.iloc[inject_pos, volz_loc] += CONFIG["INJECT_VOLZ"]

labels = pd.Series(0, index=inj_df.index)
labels.iloc[inject_pos] = 1
print(f"Injected {CONFIG['N_INJECT']} synthetic anomalies into the test window.")
print("Dates:", [d.date().isoformat() for d in inject_dates])

In [ ]:
def tolerant_scores(labels: pd.Series, flags: pd.Series, tol=CONFIG["MATCH_TOL"]):
    """Precision/recall where a positive within +/- tol positions of a true anomaly counts as a hit."""
    labels = labels.reindex(flags.index).fillna(0).astype(int).values
    pred   = flags.astype(int).values
    pos    = np.where(labels == 1)[0]

    # Recall: each injected anomaly is recovered if any flag falls within tolerance.
    recovered = 0
    for p in pos:
        lo, hi = max(0, p - tol), min(len(pred), p + tol + 1)
        if pred[lo:hi].any():
            recovered += 1
    recall = recovered / len(pos) if len(pos) else float("nan")

    # Precision: a flag is a true positive if a real anomaly sits within tolerance.
    flagged = np.where(pred == 1)[0]
    tp = 0
    for f_ in flagged:
        lo, hi = max(0, f_ - tol), min(len(labels), f_ + tol + 1)
        if labels[lo:hi].any():
            tp += 1
    precision = tp / len(flagged) if len(flagged) else float("nan")
    f1 = (2*precision*recall/(precision+recall)
          if precision and recall and not np.isnan(precision) and not np.isnan(recall) else float("nan"))
    return precision, recall, f1

# Run all three detectors on the SAME injected test window.
z_flags_inj, _   = zscore_detect(inj_df["Return"])
iso_pred_inj      = iso.predict(std_scaler.transform(inj_df[iso_feat]))
iso_flags_inj     = pd.Series(iso_pred_inj == -1, index=inj_df.index)
lstm_flags_inj, _ = lstm_flags_for(mm_scaler.transform(inj_df[lstm_feat]), inj_df.index)

rows = []
for name, fl in [("Z-Score (Tier 1)", z_flags_inj),
                 ("Isolation Forest (Tier 2)", iso_flags_inj),
                 ("LSTM Autoencoder (Tier 3)", lstm_flags_inj)]:
    p, r, f = tolerant_scores(labels, fl)
    rows.append({"Detector": name, "Precision": p, "Recall": r, "F1": f,
                 "Total flagged": int(fl.sum())})

results = pd.DataFrame(rows).set_index("Detector")
print("Validation on injected synthetic anomalies (+/-{} day tolerance):".format(CONFIG["MATCH_TOL"]))
results

**Reading the table.** The z-score often misses a fraction of injected spikes — a single huge move
*inflates its own trailing std*, so the very next comparison is desensitised. Isolation Forest, seeing the
multi-feature vector, tends to recover more. The LSTM's recall depends on how much the injection disrupts the
*sequence* it expects. The precision column is deliberately honest: some "false positives" are simply **real**
anomalies the injector didn't place — a known confound of validating unsupervised detectors on real data, and
a good point to raise rather than hide.

In [ ]:
# --- Qualitative cross-check: list the most extreme REAL flagged dates per tier ---
print("Most anomalous REAL dates to eyeball against known events")
print("(earnings, results days, the Mar-2020 COVID crash, large gap-ups, etc.):\n")

print("Top Z-Score days (|z|):")
print(z_vals_all.abs().sort_values(ascending=False).head(8).round(2).to_string(), "\n")

print("Top Isolation Forest days (most negative score = most anomalous):")
print(iso_score_all.sort_values().head(8).round(4).to_string(), "\n")

print("Top LSTM days (highest reconstruction error in test window):")
print(lstm_err_test.sort_values(ascending=False).head(8).round(6).to_string())

### Step 10 — Plot the master chart
One interactive price chart with distinct markers where each model fired:
**red dots = Z-Score, blue crosses = Isolation Forest, purple stars = LSTM.**

The Z-Score and Isolation Forest run across the full history; the **LSTM only marks the held-out test window**
(it was trained on the earlier period), which is the honest, leakage-free thing to show.

In [ ]:
def at(flags):
    """Price points where a (boolean) flag series is True, aligned to df."""
    idx = flags[flags].index
    idx = idx.intersection(df.index)
    return idx, df.loc[idx, "Close"]

z_idx,  z_y  = at(z_flags_all)
iso_idx, iso_y = at(iso_flags_all)
ls_idx, ls_y = at(lstm_flags_test)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df["Close"], mode="lines",
                         name="Close", line=dict(color="#888", width=1)))
fig.add_trace(go.Scatter(x=z_idx, y=z_y, mode="markers", name="Z-Score",
                         marker=dict(color="#e02424", size=7, symbol="circle")))
fig.add_trace(go.Scatter(x=iso_idx, y=iso_y, mode="markers", name="Isolation Forest",
                         marker=dict(color="#2563eb", size=8, symbol="x")))
fig.add_trace(go.Scatter(x=ls_idx, y=ls_y, mode="markers", name="LSTM Autoencoder",
                         marker=dict(color="#7c3aed", size=11, symbol="star")))
# shade the held-out test window
fig.add_vrect(x0=test_df.index.min(), x1=test_df.index.max(),
              fillcolor="LightGray", opacity=0.18, line_width=0,
              annotation_text="held-out test window", annotation_position="top left")

fig.update_layout(
    title=f"{CONFIG['TICKER']} — three-tier anomaly detection",
    xaxis_title="Date", yaxis_title="Close price",
    template="plotly_white", height=560, hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
)
fig.show()

### Step 11 — Agreement analysis + engineering summary
Where do the three methods agree, and where do they tell different stories? We restrict the formal comparison
to the **held-out test window**, the only region where *all three* detectors are defined — the fair common ground.

In [ ]:
common = lstm_flags_test.index  # test window, where the LSTM is defined
agree = pd.DataFrame({
    "Z":   z_flags_all.reindex(common).fillna(False),
    "ISO": iso_flags_all.reindex(common).fillna(False),
    "LSTM": lstm_flags_test.reindex(common).fillna(False),
}).astype(bool)

n_any   = (agree.any(axis=1)).sum()
n_all3  = (agree.all(axis=1)).sum()
pair = {
    "Z & ISO":  int((agree.Z & agree.ISO).sum()),
    "Z & LSTM": int((agree.Z & agree.LSTM).sum()),
    "ISO & LSTM": int((agree.ISO & agree.LSTM).sum()),
}
print(f"Test window: {len(common)} days")
print(f"Flagged by >=1 detector : {n_any}")
print(f"Flagged by ALL THREE    : {n_all3}")
print("Pairwise overlaps       :", pair)
print("\nPer-detector counts in test window:")
print(agree.sum().to_string())

In [ ]:
# Visualise per-detector counts and the unanimous set.
counts = agree.sum()
bar = go.Figure([go.Bar(
    x=["Z-Score", "Isolation Forest", "LSTM", "All three agree"],
    y=[counts["Z"], counts["ISO"], counts["LSTM"], n_all3],
    marker_color=["#e02424", "#2563eb", "#7c3aed", "#059669"],
    text=[counts["Z"], counts["ISO"], counts["LSTM"], n_all3], textposition="outside",
)])
bar.update_layout(title="Anomalies flagged per detector (held-out test window)",
                  yaxis_title="days flagged", template="plotly_white", height=420)
bar.show()

### Engineering summary — the thesis of the project

The three tiers are not redundant; they fail in **different** ways, and that is the whole point.

- **A sharp flash crash** — a single violent day — is caught **instantly by the Z-Score** at essentially zero
  compute. Neither the tree ensemble nor the deep net is needed for that.
- **A slow structural shift** — a regime change that creeps in over weeks — slips past the Z-Score (each day
  looks normal versus its immediate trailing window) but the **LSTM's memory is deep enough to notice** that the
  *sequence* no longer reconstructs.
- **Isolation Forest** sits in between: it sees the full multi-feature vector (return, volatility, volume) and
  catches multi-dimensional oddness the z-score's single dimension misses — but, scoring each day independently,
  it is **blind to temporal order**, which is precisely the gap the LSTM fills.

Where all three agree, you have a high-confidence anomaly. Where they disagree, the *reason* for the
disagreement tells you what kind of event it was — and being able to narrate that contrast is the strongest
thing you can take into an interview.

---
#### Resume bullet you can aim for
> *"Built a three-tier time-series anomaly detection pipeline (rolling z-score, Isolation Forest, LSTM
> autoencoder) on equity data; validated via synthetic anomaly injection, achieving **X% recall at Y% precision**
> under a leakage-free, time-based evaluation."*

Fill in **X / Y** from the validation table in Step 9 once you run it on your chosen ticker.